# Elliptic Bitcoin Transactions: Baselines, Boosting and the Test Evaluation

Notebook 5 of 7. It tunes the Random Forest baseline and two LightGBM models on the same temporal folds (one on raw plus graph features, one that also stacks the GraphSAGE scores), fixes every decision threshold on out-of-fold predictions, chooses the headline model between GraphSAGE and the hybrid before looking at the test period, and then scores the test period (time steps 35 to 49) exactly once.

## Setup

Imports, constants and metric helpers. Every model's threshold maximises illicit F1 on its pooled out-of-fold predictions. `SMOKE` runs 3 trials per study with smaller forests and fewer rounds; it is off in this version.

In [ ]:
import json
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from cycler import cycler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

SMOKE = False
SEED = 42
THREADS = 4
TRAIN_LAST = 34
SHUTDOWN = 43
RF_TRIALS, LGB_TRIALS, RF_TREES, MAX_ROUNDS, BOOT = (3, 3, 100, 300, 100) if SMOKE else (30, 40, 500, 2000, 1000)
TARGETS = {"auc": 0.962, "f1": 0.84, "fn_reduction": 0.22}
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
MODELS = ["rf", "raw_eng", "hybrid", "graphsage"]
MODEL_NAMES = {"rf": "Random Forest", "raw_eng": "LightGBM, raw + graph features", "hybrid": "GraphSAGE + LightGBM", "graphsage": "GraphSAGE"}
MODEL_COLORS = {"graphsage": "#2a78d6", "hybrid": "#eb6834", "raw_eng": "#1baf7a", "rf": "#eda100"}
CONFIG_LABELS = {"bce_natural": "Cross-entropy, natural ratio", "bce_balanced": "Cross-entropy, balanced seeds", "focal_natural": "Focal loss, natural ratio", "focal_balanced": "Focal loss, balanced seeds"}
STACKING = ["gnn_logit", "gnn_in_mean", "gnn_in_max", "gnn_out_mean", "gnn_out_max"]
BASE = {"objective": "binary", "learning_rate": 0.05, "metric": "average_precision", "seed": SEED, "deterministic": True, "force_col_wise": True, "num_threads": THREADS, "verbose": -1, "bagging_freq": 1}
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "text.color": INK, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2, "legend.frameon": False,
    "font.family": "sans-serif", "axes.prop_cycle": cycler(color=SERIES), "axes.titlesize": 11,
})
optuna.logging.set_verbosity(optuna.logging.WARNING)
START = time.time()


def find_file(name):
    hits = sorted(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def elapsed():
    return f"elapsed {(time.time() - START) / 60:.1f} minutes"


def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-np.asarray(values, dtype=float)))


def confusion(y, p, t):
    pred = p >= t
    tp = int((pred & (y == 1)).sum())
    fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum())
    tn = int((~pred & (y == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": precision, "recall": recall, "f1": f1}


def metrics_at(y, p, t):
    return {"auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p)), **confusion(y, p, t), "threshold": float(t)}


def best_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    i = int(np.nanargmax(f1))
    return float(thresholds[i]), float(f1[i])


def matched_threshold(y, p, target):
    precision, recall, thresholds = precision_recall_curve(y, p)
    valid = np.flatnonzero(precision[:-1] >= target)
    return float(thresholds[valid[0]]) if len(valid) else 1.0


def bootstrap(y, p, t, reps):
    rng = np.random.default_rng(SEED)
    stats = {k: [] for k in ("auc", "pr_auc", "f1", "precision", "recall")}
    for _ in range(reps):
        idx = rng.integers(0, len(y), len(y))
        m = metrics_at(y[idx], p[idx], t)
        for k in stats:
            stats[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in stats.items()}


def reduction_ci(y, p_rf, t_rf, p_model, t_model, reps):
    rng = np.random.default_rng(SEED)
    values = []
    for _ in range(reps):
        idx = rng.integers(0, len(y), len(y))
        positive = y[idx] == 1
        fn_rf = int(((p_rf[idx] < t_rf) & positive).sum())
        fn_model = int(((p_model[idx] < t_model) & positive).sum())
        values.append((fn_rf - fn_model) / fn_rf if fn_rf else 0.0)
    return [float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))]


print(f"smoke={SMOKE}, rf trials {RF_TRIALS}, lightgbm trials {LGB_TRIALS}, rf trees {RF_TREES}, bootstrap {BOOT}")
print(f"lightgbm {lgb.__version__}, optuna {optuna.__version__}, pandas {pd.__version__}")

**Conclusion.** The full run gives the Random Forest 30 trials of 500 trees and each LightGBM model 40 trials, with 1,000 bootstrap resamples for every test interval, on Kaggle's LightGBM 4.6.0 and Optuna 5.0.0.

## Assemble the tables

Aim: join raw features, engineered features, GraphSAGE stacking features and the ablation logits, then separate the labelled training period from the test period. The three feature sets are the Random Forest baseline's (165 raw, as in the published baseline), LightGBM A's (raw plus engineered) and the hybrid's (raw plus engineered plus stacking).

In [ ]:
feat = pd.read_parquet(find_file("features.parquet"))
split = pd.read_parquet(find_file("split.parquet"))
gnn = pd.read_parquet(find_file("gnn_scores.parquet"))
abl = pd.read_parquet(find_file("ablation_scores.parquet"))
meta = json.loads(find_file("feature_meta.json").read_text(encoding="utf-8"))
RAW, ENG = meta["raw"], meta["engineered"]
CONFIGS = [c for c in abl.columns if c not in ("txId", "set")]
data = feat.merge(split[["txId", "temporal_fold"]], on="txId").merge(gnn[["txId"] + STACKING], on="txId").merge(abl[["txId"] + CONFIGS], on="txId")
train = data[(data.time_step <= TRAIN_LAST) & (data.label >= 0)].reset_index(drop=True)
test_all = data[data.time_step > TRAIN_LAST].reset_index(drop=True)
y = train.label.to_numpy().astype(int)
folds = train.temporal_fold.to_numpy()
lab = test_all.label.to_numpy() >= 0
y_test = test_all.label.to_numpy()[lab].astype(int)
steps_test = test_all.time_step.to_numpy()[lab]
SETS = {"rf": RAW, "raw_eng": RAW + ENG, "hybrid": RAW + ENG + STACKING}
X_RF = train[RAW].to_numpy(np.float32)
print(f"training rows {len(train):,} (illicit {int(y.sum()):,}), test nodes {len(test_all):,}, labelled test {int(lab.sum()):,} (illicit {int(y_test.sum()):,})")
print({name: len(cols) for name, cols in SETS.items()})

**Conclusion.** Training uses all 29,894 labelled transactions of steps 1 to 34 (3,462 illicit). The test period holds 67,504 transactions, of which 16,670 are labelled and 1,083 illicit; every one of the 67,504 gets a prediction so the serving app can show scores for unlabelled transactions too. The three feature sets have 165 (Random Forest), 342 (LightGBM A) and 347 columns (the hybrid).

## Random Forest baseline

Aim: give the baseline the same treatment as the challengers, a bounded Optuna search (30 trials, 500 trees) on the same five temporal folds with the same objective (pooled out-of-fold PR-AUC). A weak, untuned baseline would make any false-negative reduction meaningless.

In [ ]:
def rf_oof(params, trial=None):
    pred = np.zeros(len(train))
    fold_ap = []
    for k in range(5):
        fit_rows = folds != k
        model = RandomForestClassifier(n_estimators=RF_TREES, n_jobs=THREADS, random_state=SEED, **params)
        model.fit(X_RF[fit_rows], y[fit_rows])
        pred[~fit_rows] = model.predict_proba(X_RF[~fit_rows])[:, 1]
        fold_ap.append(average_precision_score(y[~fit_rows], pred[~fit_rows]))
        if trial is not None:
            trial.report(float(np.mean(fold_ap)), k)
            if trial.should_prune():
                print(f"rf trial {trial.number} pruned after fold {k}")
                raise optuna.TrialPruned()
    return pred


def rf_objective(trial):
    params = {
        "max_features": trial.suggest_float("max_features", 0.05, 0.5),
        "max_depth": trial.suggest_categorical("max_depth", [None, 10, 15, 20, 30, 40]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
    }
    t0 = time.time()
    value = average_precision_score(y, rf_oof(params, trial))
    print(f"rf trial {trial.number}: pr_auc {value:.4f} | {params}")
    print(f"rf trial {trial.number} elapsed {(time.time() - t0) / 60:.1f} minutes")
    return value


def make_study():
    return optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED), pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))


rf_study = make_study()
rf_study.optimize(rf_objective, n_trials=RF_TRIALS)
BEST = {"rf": dict(rf_study.best_params)}
print(f"best rf pr_auc {rf_study.best_value:.4f} with {BEST['rf']}")
print(elapsed())

**Conclusion.** Tuning took the baseline seriously: the best Random Forest reaches a pooled out-of-fold PR-AUC of 0.9804 with 11% of features per split, depth 40, at least 3 samples per leaf and per-tree class balancing. This is a strong baseline, so any gain over it has to be real.

## LightGBM studies

Aim: tune LightGBM A (raw plus engineered) and the hybrid (plus stacking) with 40 trials each at learning rate 0.05, early-stopping on the held-out fold inside tuning only. The mean best iteration across folds becomes each model's fixed round count. The gap between the two studies is the first measure of what GraphSAGE adds.

In [ ]:
def lgb_cv(cols, params, rounds=None, trial=None, label=""):
    pred = np.zeros(len(train))
    iters = []
    fold_ap = []
    for k in range(5):
        fit_rows = folds != k
        dtrain = lgb.Dataset(train.loc[fit_rows, cols], y[fit_rows], free_raw_data=False)
        if rounds is None:
            dvalid = lgb.Dataset(train.loc[~fit_rows, cols], y[~fit_rows], reference=dtrain)
            booster = lgb.train({**BASE, **params}, dtrain, num_boost_round=MAX_ROUNDS, valid_sets=[dvalid], callbacks=[lgb.early_stopping(100, verbose=False)])
            iters.append(booster.best_iteration)
            p = booster.predict(train.loc[~fit_rows, cols], num_iteration=booster.best_iteration)
        else:
            booster = lgb.train({**BASE, **params}, dtrain, num_boost_round=rounds)
            p = booster.predict(train.loc[~fit_rows, cols])
        pred[~fit_rows] = p
        fold_ap.append(average_precision_score(y[~fit_rows], p))
        if trial is not None:
            trial.report(float(np.mean(fold_ap)), k)
            if trial.should_prune():
                print(f"{label} trial {trial.number} pruned after fold {k}")
                raise optuna.TrialPruned()
    return pred, iters


def lgb_objective(name):
    def objective(trial):
        params = {
            "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 200, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 20.0, log=True),
        }
        t0 = time.time()
        pred, iters = lgb_cv(SETS[name], params, trial=trial, label=name)
        value = average_precision_score(y, pred)
        rounds = int(round(np.mean(iters)))
        trial.set_user_attr("rounds", rounds)
        print(f"{name} trial {trial.number}: pr_auc {value:.4f} rounds {rounds} | {params}")
        print(f"{name} trial {trial.number} elapsed {(time.time() - t0) / 60:.1f} minutes")
        return value
    return objective


for name in ("raw_eng", "hybrid"):
    study = make_study()
    study.optimize(lgb_objective(name), n_trials=LGB_TRIALS)
    BEST[name] = {"params": dict(study.best_params), "rounds": int(study.best_trial.user_attrs["rounds"])}
    print(f"best {name} pr_auc {study.best_value:.4f}, rounds {BEST[name]['rounds']}, params {BEST[name]['params']}")
print(elapsed())

**Conclusion.** LightGBM A reaches a pooled out-of-fold PR-AUC of 0.9888 (446 rounds) and the hybrid 0.9879 (412 rounds). On the training period the GraphSAGE stacking features add nothing over raw plus engineered features; the hybrid's tuned settings differ mainly in sampling only 36% of features per tree and weighting illicit transactions 3.4 times. Whether GraphSAGE helps on unseen data is answered only by the test period.

## Out-of-fold predictions and thresholds

Aim: produce clean out-of-fold predictions for every model (the LightGBM folds refit at the fixed round count, so no prediction early-stops on its own fold), set each model's F1-optimal threshold, set matched-precision thresholds equal to the Random Forest's out-of-fold precision, and choose the headline model between GraphSAGE and the hybrid by out-of-fold F1, all before any test prediction exists.

In [ ]:
OOF = {"rf": rf_oof(BEST["rf"])}
for name in ("raw_eng", "hybrid"):
    OOF[name], _ = lgb_cv(SETS[name], BEST[name]["params"], rounds=BEST[name]["rounds"])
OOF["graphsage"] = sigmoid(train.gnn_logit.to_numpy())
for config in CONFIGS:
    OOF[config] = sigmoid(train[config].to_numpy())
THRESH = {}
OOF_METRICS = {}
for name, p in OOF.items():
    THRESH[name], _ = best_threshold(y, p)
    OOF_METRICS[name] = metrics_at(y, p, THRESH[name])
    m = OOF_METRICS[name]
    print(f"{name}: oof auc {m['auc']:.4f}, pr_auc {m['pr_auc']:.4f}, f1 {m['f1']:.4f}, precision {m['precision']:.4f}, recall {m['recall']:.4f}, threshold {m['threshold']:.4f}")
RF_PRECISION = OOF_METRICS["rf"]["precision"]
MATCHED = {name: matched_threshold(y, OOF[name], RF_PRECISION) for name in ("raw_eng", "hybrid", "graphsage")}
print(f"random forest out-of-fold precision {RF_PRECISION:.4f}; matched thresholds {({k: round(v, 4) for k, v in MATCHED.items()})}")
HEADLINE = "hybrid" if OOF_METRICS["hybrid"]["f1"] >= OOF_METRICS["graphsage"]["f1"] else "graphsage"
print(f"headline model chosen on out-of-fold F1: {HEADLINE}")

**Conclusion.** Out of fold, LightGBM A has the highest F1 (0.960), then the hybrid (0.956), the Random Forest (0.939) and GraphSAGE alone (0.896). The headline is chosen between GraphSAGE and the hybrid, as fixed in the spec, and the hybrid wins on out-of-fold F1 before any test prediction exists. Every threshold is now fixed; the Random Forest's out-of-fold precision of 0.942 sets the matched-precision thresholds.

## Final fits

Aim: refit Random Forest, LightGBM A and the hybrid on all labelled training rows (the hybrid's training rows carry out-of-fold stacking features, its test rows the 15-model averages) and predict every test-period node, labelled or not. No metric is computed in this cell.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=RF_TREES, n_jobs=THREADS, random_state=SEED, **BEST["rf"]).fit(X_RF, y)
TEST_ALL = {"rf": rf_model.predict_proba(test_all[RAW].to_numpy(np.float32))[:, 1]}
boosters = {}
for name in ("raw_eng", "hybrid"):
    boosters[name] = lgb.train({**BASE, **BEST[name]["params"]}, lgb.Dataset(train[SETS[name]], y), num_boost_round=BEST[name]["rounds"])
    TEST_ALL[name] = boosters[name].predict(test_all[SETS[name]])
TEST_ALL["graphsage"] = sigmoid(test_all.gnn_logit.to_numpy())
for config in CONFIGS:
    TEST_ALL[config] = sigmoid(test_all[config].to_numpy())
TEST = {name: p[lab] for name, p in TEST_ALL.items()}
print(f"test predictions for {len(TEST_ALL)} models over {len(test_all):,} nodes")
print(elapsed())

**Conclusion.** All final models are refit on every labelled training transaction and have scored all 67,504 test-period transactions. No metric has been computed yet.

## The test evaluation

Aim: score the held-out period once, at the thresholds fixed above: AUC-ROC, PR-AUC, illicit F1, precision, recall and the false-negative count for every model and ablation configuration, each with a bootstrap 95% interval; the false-negative reduction against the Random Forest at each model's own threshold and at matched precision; per-step results and the split at the step 43 dark-market shutdown; and the benchmark table.

In [ ]:
results = {}
for name in MODELS + CONFIGS:
    m = metrics_at(y_test, TEST[name], THRESH[name])
    m["ci"] = bootstrap(y_test, TEST[name], THRESH[name], BOOT)
    results[name] = m
    print(f"{name}: auc {m['auc']:.4f} [{m['ci']['auc'][0]:.4f}, {m['ci']['auc'][1]:.4f}], pr_auc {m['pr_auc']:.4f}, f1 {m['f1']:.4f} [{m['ci']['f1'][0]:.4f}, {m['ci']['f1'][1]:.4f}], precision {m['precision']:.4f}, recall {m['recall']:.4f}, tp {m['tp']}, fp {m['fp']}, fn {m['fn']}")
fn_rf = results["rf"]["fn"]
matched = {name: metrics_at(y_test, TEST[name], MATCHED[name]) for name in MATCHED}
reduction_own = {name: (fn_rf - results[name]["fn"]) / fn_rf for name in MODELS if name != "rf"}
reduction_matched = {name: (fn_rf - matched[name]["fn"]) / fn_rf for name in MATCHED}
ci_own = reduction_ci(y_test, TEST["rf"], THRESH["rf"], TEST[HEADLINE], THRESH[HEADLINE], BOOT)
ci_matched = reduction_ci(y_test, TEST["rf"], THRESH["rf"], TEST[HEADLINE], MATCHED[HEADLINE], BOOT)
for name in reduction_own:
    print(f"false negatives vs random forest, {name}: own threshold {reduction_own[name]:+.2%} (fn {results[name]['fn']} vs {fn_rf}), matched precision {reduction_matched[name]:+.2%} (fn {matched[name]['fn']}, precision {matched[name]['precision']:.4f})")
print(f"headline reduction interval: own {ci_own[0]:+.2%} to {ci_own[1]:+.2%}, matched {ci_matched[0]:+.2%} to {ci_matched[1]:+.2%}")
steps = sorted(int(s) for s in np.unique(steps_test))
per_step = {"steps": steps, "illicit": [int(y_test[steps_test == s].sum()) for s in steps], "f1": {}, "recall": {}}
for name in MODELS:
    rows = [confusion(y_test[steps_test == s], TEST[name][steps_test == s], THRESH[name]) for s in steps]
    per_step["f1"][name] = [r["f1"] for r in rows]
    per_step["recall"][name] = [r["recall"] for r in rows]
print(pd.DataFrame({"illicit": per_step["illicit"], **{f"f1_{n}": per_step["f1"][n] for n in MODELS}}, index=steps).round(3).to_string())
early = steps_test < SHUTDOWN
shutdown = {name: {"before": confusion(y_test[early], TEST[name][early], THRESH[name])["f1"], "after": confusion(y_test[~early], TEST[name][~early], THRESH[name])["f1"]} for name in MODELS}
for name in MODELS:
    print(f"{name}: f1 steps 35-42 {shutdown[name]['before']:.4f}, steps 43-49 {shutdown[name]['after']:.4f}")
ablation = [{"config": c, "label": CONFIG_LABELS[c], "oof_pr_auc": OOF_METRICS[c]["pr_auc"], "oof_f1": OOF_METRICS[c]["f1"], "test_pr_auc": results[c]["pr_auc"], "test_f1": results[c]["f1"], "test_recall": results[c]["recall"]} for c in CONFIGS]
print(pd.DataFrame(ablation).set_index("config").round(4).to_string())
head = results[HEADLINE]
benchmarks = [
    {"name": "AUC-ROC", "target": TARGETS["auc"], "achieved": head["auc"], "low": head["ci"]["auc"][0], "high": head["ci"]["auc"][1], "met": head["auc"] >= TARGETS["auc"]},
    {"name": "Illicit F1", "target": TARGETS["f1"], "achieved": head["f1"], "low": head["ci"]["f1"][0], "high": head["ci"]["f1"][1], "met": head["f1"] >= TARGETS["f1"]},
    {"name": "Fewer false negatives than Random Forest, own threshold", "target": TARGETS["fn_reduction"], "achieved": reduction_own[HEADLINE], "low": ci_own[0], "high": ci_own[1], "met": reduction_own[HEADLINE] >= TARGETS["fn_reduction"]},
    {"name": "Fewer false negatives than Random Forest, matched precision", "target": TARGETS["fn_reduction"], "achieved": reduction_matched[HEADLINE], "low": ci_matched[0], "high": ci_matched[1], "met": reduction_matched[HEADLINE] >= TARGETS["fn_reduction"]},
]
print(f"headline model: {HEADLINE} ({MODEL_NAMES[HEADLINE]})")
print(pd.DataFrame(benchmarks).round(4).to_string(index=False))
metrics = {
    "headline": HEADLINE, "model_names": MODEL_NAMES, "thresholds": THRESH, "matched_thresholds": MATCHED,
    "rf_oof_precision": RF_PRECISION, "oof": OOF_METRICS, "test": results, "matched": matched,
    "fn_reduction": {"own": reduction_own, "matched": reduction_matched, "headline_ci": {"own": ci_own, "matched": ci_matched}},
    "per_step": per_step, "shutdown": shutdown, "ablation": ablation, "benchmarks": benchmarks,
    "counts": {"train_labelled": int(len(train)), "train_illicit": int(y.sum()), "test_nodes": int(len(test_all)), "test_labelled": int(lab.sum()), "test_illicit": int(y_test.sum())},
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding="utf-8")

**Conclusion.** This is the only look at the test period, and none of the four benchmarks is met. The hybrid reaches an AUC-ROC of 0.950 [0.944, 0.956] against the 0.962 target and an illicit F1 of 0.819 [0.800, 0.837] against 0.84, so the target lies just above the upper end of the F1 interval. It misses more illicit transactions than the Random Forest, not fewer: 307 against 281 (9.3% more), and at matched precision the two are statistically tied (1.8% more, interval -4.4% to +0.4%). The hybrid is nonetheless the best model overall: against the Random Forest it lifts F1 from 0.690 to 0.819 and precision from 0.646 to 0.955, cutting false positives from 440 to 37, because the Random Forest's out-of-fold threshold flags far more transactions once the test period drifts. GraphSAGE stacking helps slightly on unseen data (hybrid against LightGBM A: AUC 0.950 against 0.946, F1 0.819 against 0.803), while GraphSAGE alone is clearly weaker (F1 0.614, AUC 0.892), close to the published 0.689. The dark-market shutdown dominates everything: on steps 35 to 42 the hybrid's F1 is 0.905, on steps 43 to 49 it is 0.032, and every other model collapses the same way. On the test period the imbalance ablation keeps its ordering at the top: focal loss with balanced seeds is best (PR-AUC 0.667, F1 0.628 against 0.633 and 0.578 for plain cross-entropy), focal loss alone helps a little and balanced sampling alone hurts.

## Figures

Aim: plot the test-period ROC and precision-recall curves for the four models, F1 by time step with the shutdown marked, false negatives at each model's threshold, and the ablation, with the numbers behind each printed above.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 5))
for name in MODELS:
    fpr, tpr, _ = roc_curve(y_test, TEST[name])
    ax.plot(fpr, tpr, color=MODEL_COLORS[name], label=f"{MODEL_NAMES[name]} ({results[name]['auc']:.3f})")
ax.plot([0, 1], [0, 1], color=MUTED, linestyle="--", linewidth=1)
ax.set(xlabel="false positive rate", ylabel="true positive rate", title="ROC, test steps 35-49")
ax.legend(loc="lower right", fontsize=8)
save(fig, "roc_test")
fig, ax = plt.subplots(figsize=(5.6, 5))
for name in MODELS:
    precision, recall, _ = precision_recall_curve(y_test, TEST[name])
    ax.plot(recall, precision, color=MODEL_COLORS[name], label=f"{MODEL_NAMES[name]} ({results[name]['pr_auc']:.3f})")
ax.axhline(y_test.mean(), color=MUTED, linestyle="--", linewidth=1)
ax.set(xlabel="recall", ylabel="precision", title="Precision-recall, test steps 35-49")
ax.legend(loc="lower left", fontsize=8)
save(fig, "pr_test")
fig, ax = plt.subplots(figsize=(9, 3.8))
for name in MODELS:
    ax.plot(steps, per_step["f1"][name], color=MODEL_COLORS[name], marker="o", markersize=5, label=MODEL_NAMES[name])
ax.axvline(SHUTDOWN - 0.5, color=MUTED, linestyle="--", linewidth=1)
ax.text(SHUTDOWN - 0.3, 0.95, "dark-market shutdown", color=INK_2, va="top", fontsize=8)
ax.set(xlabel="time step", ylabel="illicit F1", ylim=(0, 1), title="Illicit F1 by test time step")
ax.legend(fontsize=8, ncol=2)
save(fig, "f1_by_step")
fig, ax = plt.subplots(figsize=(6.4, 3.2))
fns = [results[n]["fn"] for n in MODELS]
ax.barh([MODEL_NAMES[n] for n in MODELS], fns, color=[MODEL_COLORS[n] for n in MODELS], height=0.6)
for i, value in enumerate(fns):
    ax.text(value, i, f" {value}", va="center", color=INK_2, fontsize=9)
ax.set(xlabel="false negatives on the test period (own thresholds)", title="Missed illicit transactions")
save(fig, "false_negatives")
fig, ax = plt.subplots(figsize=(6.4, 3.2))
values = [results[c]["pr_auc"] for c in CONFIGS]
ax.barh([CONFIG_LABELS[c] for c in CONFIGS], values, color=SERIES[0], height=0.6)
for i, value in enumerate(values):
    ax.text(value, i, f" {value:.3f}", va="center", color=INK_2, fontsize=9)
ax.set(xlabel="test PR-AUC (GraphSAGE, one seed)", title="Imbalance ablation on the test period")
save(fig, "ablation_test")

**Conclusion.** The figures show the same picture. The ROC curves of the three tree-based models nearly coincide, with the hybrid on top at 0.950 and GraphSAGE alone well below. In precision-recall, all three tree models hold precision near 1.0 up to about 70% recall and then fall steeply together, which is the shutdown period's illicit transactions they cannot find. Per step, the hybrid and LightGBM A track each other above 0.75 until step 42 and every model drops to near zero at step 43. The false-negative bars put the Random Forest lowest (281) and GraphSAGE alone highest (536). In the ablation chart, focal loss with balanced seeds is the clear leader.

## Write the outputs

Aim: save the boosters, tuned settings, thresholds and predictions that notebooks 6 and 7 consume.

In [ ]:
boosters["hybrid"].save_model(str(OUT / "lgbm_hybrid.txt"))
boosters["raw_eng"].save_model(str(OUT / "lgbm_raw_eng.txt"))
(OUT / "best_params_tabular.json").write_text(json.dumps(BEST, indent=2), encoding="utf-8")
(OUT / "thresholds.json").write_text(json.dumps({**THRESH, "matched": MATCHED, "headline": HEADLINE}, indent=2), encoding="utf-8")
oof_frame = train[["txId", "label", "temporal_fold"]].copy()
for name, p in OOF.items():
    oof_frame[name] = p.astype(np.float32)
oof_frame.to_parquet(OUT / "oof.parquet", index=False)
test_frame = test_all[["txId", "time_step", "label"]].copy()
for name, p in TEST_ALL.items():
    test_frame[name] = p.astype(np.float32)
test_frame.to_parquet(OUT / "test_pred.parquet", index=False)
for path in sorted(OUT.glob("*.*")):
    print(f"{path.name}: {path.stat().st_size / 1e6:.2f} MB")
print(elapsed())

**Conclusion.** The boosters, tuned settings, thresholds, out-of-fold predictions and test predictions for all 67,504 test-period transactions are saved for notebooks 6 and 7. metrics.json holds every number above, including the benchmark table with its intervals.